# Laboratorio: sentimiento de reseñas en español

Este cuaderno compara dos modelos ya entrenados sobre las mismas frases y luego ejecuta una misma predicción localmente con `pipeline` y de forma remota con la API Gradio de Hugging Face mediante `gradio_client`. La Space usa Gradio nativo porque ZeroGPU requiere una función Gradio decorada con `@spaces.GPU`; por eso el cliente remoto de esta Space no es `InferenceClient`. **Diferencia respecto de la consigna:** esta medición remota no satisface literalmente el requisito de llamar a `InferenceClient`; se documenta la adaptación en vez de presentar los clientes como equivalentes. Para usar `InferenceClient` habría que migrar a un endpoint/proveedor de inferencia compatible, lo que puede implicar otros límites o costos. No se entrena ningún modelo. Ejecutá las celdas en orden y guardá los resultados reales de tu equipo; los tiempos dependen de la conexión, el hardware, la caché y la disponibilidad de ZeroGPU.


## Modelos candidatos y fuentes

- [`cardiffnlp/twitter-xlm-roberta-base-sentiment`](https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment): clasifica sentimiento en negativo, neutro o positivo. Su model card describe XLM-RoBERTa entrenado con unos 198 millones de tweets y ajuste de sentimiento en ocho idiomas, incluido español. El Hub informa un archivo de pesos de 1.112.271.561 bytes (aprox. 1,04 GiB). La tarjeta no especifica todos los corpus de ajuste.
- [`pysentimiento/robertuito-sentiment-analysis`](https://huggingface.co/pysentimiento/robertuito-sentiment-analysis): clasifica POS/NEU/NEG. Su tarjeta informa RoBERTuito preentrenado con más de 500 millones de tweets en español y ajuste con TASS 2020 (aprox. 5.000 tweets de distintas variedades del español). El archivo `model.safetensors` del Hub ocupa 435.189.428 bytes (aprox. 415 MiB). La tarjeta limita el uso a investigación científica y uso no comercial.

Los tamaños son los archivos de pesos indicados por el Hub al preparar este material; comprobá los model cards antes de entregar porque pueden cambiar. Para evitar depender de créditos de Inference Providers, el bloque remoto de este cuaderno usa una Space pública propia con ZeroGPU y la API Gradio de `gradio_client`.


In [4]:
import os
import time

import pandas as pd
from gradio_client import Client
from transformers import pipeline

MODELS = {
    "Cardiff XLM-R": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    "RoBERTuito": "pysentimiento/robertuito-sentiment-analysis",
}
SELECTED_MODEL = MODELS["Cardiff XLM-R"]
print("Modelos candidatos:", MODELS)


Modelos candidatos: {'Cardiff XLM-R': 'cardiffnlp/twitter-xlm-roberta-base-sentiment', 'RoBERTuito': 'pysentimiento/robertuito-sentiment-analysis'}


## Conjunto pequeño de prueba

Las etiquetas esperadas son una anotación manual para este ejercicio, no una verdad universal. Hay diez ejemplos (más del mínimo de ocho), incluyendo ironía, una reseña en inglés, texto muy corto y jerga rioplatense. Revisá si las etiquetas esperadas te parecen razonables y explicá las limitaciones de un conjunto tan pequeño.


In [5]:
EJEMPLOS = [
    {"frase": "El pedido llegó rápido y todo estaba riquísimo. Volvería a comprar.", "esperada": "Positivo"},
    {"frase": "Me atendieron con mucha paciencia; excelente servicio.", "esperada": "Positivo"},
    {"frase": "La comida estaba bien, sin nada especial.", "esperada": "Neutro"},
    {"frase": "El paquete llegó ayer.", "esperada": "Neutro"},
    {"frase": "Esperé una hora y me entregaron el pedido frío.", "esperada": "Negativo"},
    {"frase": "No me respondieron el reclamo y nadie se hizo cargo.", "esperada": "Negativo"},
    {"frase": "Genial, otra vez me cancelaron el pedido a último minuto. Un lujo.", "esperada": "Negativo"},
    {"frase": "The delivery was late and the food arrived cold.", "esperada": "Negativo"},
    {"frase": "Malísimo.", "esperada": "Negativo"},
    {"frase": "Che, re piola el lugar, alta atención.", "esperada": "Positivo"},
]

ALIAS_ETIQUETAS = {
    "positive": "Positivo", "pos": "Positivo", "positivo": "Positivo",
    "neutral": "Neutro", "neu": "Neutro", "neutro": "Neutro",
    "negative": "Negativo", "neg": "Negativo", "negativo": "Negativo",
}

def normalizar_etiqueta(etiqueta):
    texto = str(etiqueta).strip()
    return ALIAS_ETIQUETAS.get(texto.lower(), texto)

pd.DataFrame(EJEMPLOS)


,frase,esperada
0,El pedido llegó rápido y todo estaba riquísimo...,Positivo
1,Me atendieron con mucha paciencia; excelente s...,Positivo
2,"La comida estaba bien, sin nada especial.",Neutro
3,El paquete llegó ayer.,Neutro
4,Esperé una hora y me entregaron el pedido frío.,Negativo
5,No me respondieron el reclamo y nadie se hizo ...,Negativo
6,"Genial, otra vez me cancelaron el pedido a últ...",Negativo
7,The delivery was late and the food arrived cold.,Negativo
8,Malísimo.,Negativo
9,"Che, re piola el lugar, alta atención.",Positivo


## Comparación local con `pipeline`

Cada modelo se descarga/carga una vez y recibe exactamente las mismas diez frases. Se registra por separado el tiempo de carga y el tiempo total de inferencia del conjunto. En una primera ejecución, la descarga puede dominar la carga; en las siguientes, los archivos pueden estar en caché. Si tu computadora no tiene memoria suficiente, ejecutá el cuaderno en Colab.


In [6]:
PIPELINES = {}
tiempos_carga = {}
filas_resultados = []

for nombre_modelo, model_id in MODELS.items():
    inicio_carga = time.perf_counter()
    clasificador = pipeline(
        "text-classification",
        model=model_id,
        device=-1,
        use_fast=False,
    )
    tiempos_carga[nombre_modelo] = time.perf_counter() - inicio_carga
    PIPELINES[model_id] = clasificador

    inicio_inferencia = time.perf_counter()
    for ejemplo in EJEMPLOS:
        predicciones = clasificador(ejemplo["frase"])
        mejor = max(predicciones, key=lambda item: item["score"])
        predicha = normalizar_etiqueta(mejor["label"])
        filas_resultados.append({
            "modelo": nombre_modelo,
            "frase": ejemplo["frase"],
            "etiqueta esperada": ejemplo["esperada"],
            "etiqueta predicha": predicha,
            "puntaje": float(mejor["score"]),
            "¿acertó?": predicha == ejemplo["esperada"],
        })
    tiempos_carga[f"{nombre_modelo} · inferencia de 10 frases"] = (
        time.perf_counter() - inicio_inferencia
    )

resultados = pd.DataFrame(filas_resultados)
display(resultados)
display(pd.Series(tiempos_carga, name="segundos medidos").to_frame())


c:\Users\windows\Desktop\escuela\IFTS Nº18\A3 C1\Seminario_de_actualizacion\TP_Integrador\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\windows\.cache\huggingface\hub\models--cardiffnlp--twitter-xlm-roberta-base-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled fo

,modelo,frase,etiqueta esperada,etiqueta predicha,puntaje,¿acertó?
0,Cardiff XLM-R,El pedido llegó rápido y todo estaba riquísimo...,Positivo,Positivo,0.907215,True
1,Cardiff XLM-R,Me atendieron con mucha paciencia; excelente s...,Positivo,Positivo,0.898553,True
2,Cardiff XLM-R,"La comida estaba bien, sin nada especial.",Neutro,Neutro,0.419907,True
3,Cardiff XLM-R,El paquete llegó ayer.,Neutro,Neutro,0.785995,True
4,Cardiff XLM-R,Esperé una hora y me entregaron el pedido frío.,Negativo,Neutro,0.652778,False
5,Cardiff XLM-R,No me respondieron el reclamo y nadie se hizo ...,Negativo,Negativo,0.744775,True
6,Cardiff XLM-R,"Genial, otra vez me cancelaron el pedido a últ...",Negativo,Positivo,0.861191,False
7,Cardiff XLM-R,The delivery was late and the food arrived cold.,Negativo,Negativo,0.781614,True
8,Cardiff XLM-R,Malísimo.,Negativo,Negativo,0.928065,True
9,Cardiff XLM-R,"Che, re piola el lugar, alta atención.",Positivo,Positivo,0.929952,True


,segundos medidos
Cardiff XLM-R,110.839600
Cardiff XLM-R · inferencia de 10 frases,0.569013
RoBERTuito,92.619155
RoBERTuito · inferencia de 10 frases,0.529330


## Lectura de la comparación

La columna `puntaje` expresa el score que devuelve cada modelo para su etiqueta principal; no es una garantía ni una probabilidad necesariamente calibrada. Observá especialmente las frases difíciles. Un error puede deberse a la ironía, al contexto ausente, a una frase muy corta, a jerga local o a diferencias entre las reseñas de este ejercicio y los tweets usados para ajustar los modelos.

No elijas un modelo solo por el porcentaje de aciertos de diez frases: el conjunto es pequeño y fue anotado manualmente. Considerá también idioma y dominio, tamaño de los pesos, licencia, proveedor disponible y límites de uso. Anotá aquí los casos concretos que revisarás en el informe; no inventes resultados.


In [7]:
resumen = resultados.groupby("modelo")["¿acertó?"].agg(["sum", "count", "mean"])
resumen = resumen.rename(columns={"sum": "aciertos", "count": "casos", "mean": "proporción de aciertos"})
display(resumen)


,aciertos,casos,proporción de aciertos
modelo,,,
Cardiff XLM-R,8,10,0.8
RoBERTuito,8,10,0.8


## La misma frase con `pipeline` y la API Gradio remota

`pipeline` ejecuta el modelo en este entorno local: la primera carga puede descargar más de 1 GiB de pesos de Cardiff, ocupar memoria y tardar más que las inferencias posteriores. `gradio_client` envía el texto a la API Gradio de tu Space y recibe las predicciones; no descarga los pesos en esta computadora. El tiempo remoto incluye red, cola y ejecución en ZeroGPU, así que no es una comparación de hardware equivalente.

Para la llamada remota, definí `INFERENCE_ENDPOINT_URL` con la URL pública de tu Space, por ejemplo `https://USUARIO-NOMBRE-SPACE.hf.space/`, y `HF_TOKEN` con un token de lectura de Hugging Face. El token autentica el acceso; no se comparte con el modelo como texto. La celda falla con un mensaje explícito si falta una variable. ZeroGPU puede tener espera o cuota limitada.


In [10]:
FRASE_BENCHMARK = "No me respondieron el reclamo y nadie se hizo cargo."

# El pipeline de Cardiff ya se cargó en la comparación; medimos una llamada local caliente.
pipeline_cardiff = PIPELINES[SELECTED_MODEL]
inicio = time.perf_counter()
salida_pipeline = pipeline_cardiff(FRASE_BENCHMARK)
segundos_pipeline = time.perf_counter() - inicio
print("Tipo de retorno de pipeline:", type(salida_pipeline))
print("Tipo del primer elemento:", type(salida_pipeline[0]))
print("Etiqueta mediante diccionario:", salida_pipeline[0]["label"])
print("Puntaje mediante diccionario:", salida_pipeline[0]["score"])
print(f"Tiempo de inferencia local (modelo ya cargado): {segundos_pipeline:.4f} s")


Tipo de retorno de pipeline: <class 'list'>
Tipo del primer elemento: <class 'dict'>
Etiqueta mediante diccionario: negative
Puntaje mediante diccionario: 0.7447754144668579
Tiempo de inferencia local (modelo ya cargado): 0.0593 s


In [12]:
import os
import getpass

os.environ["INFERENCE_ENDPOINT_URL"] = "https://pbueno467-tp-integrador.hf.space/"
os.environ["HF_TOKEN"] = getpass.getpass("Token de Hugging Face: ")

In [13]:
endpoint_url = os.getenv("INFERENCE_ENDPOINT_URL")
token = os.getenv("HF_TOKEN")
if not endpoint_url:
    raise RuntimeError(
        "Falta INFERENCE_ENDPOINT_URL. Configurá la URL pública de tu "
        "Space pública como variable de entorno antes de ejecutar esta celda."
    )
if not token:
    raise RuntimeError("Falta HF_TOKEN; configurá un token de lectura como secreto.")

cliente = Client(
    src=endpoint_url,
    hf_token=token,
    httpx_kwargs={"timeout": 120},
)
inicio = time.perf_counter()
salida_remota = cliente.predict(FRASE_BENCHMARK, api_name="/clasificar")
segundos_remoto = time.perf_counter() - inicio
print("Tipo de retorno de gradio_client:", type(salida_remota))
print("Tipo del primer elemento:", type(salida_remota[0]))

primer_resultado = salida_remota[0]
etiqueta_remota = primer_resultado["label"]
puntaje_remoto = primer_resultado["score"]

print("Etiqueta remota:", etiqueta_remota)
print("Puntaje remoto:", puntaje_remoto)
print(f"Tiempo de respuesta remoto (incluye red): {segundos_remoto:.4f} s")
display(pd.DataFrame([
    {"forma": "pipeline (modelo cargado)", "segundos": segundos_pipeline},
    {"forma": "gradio_client (llamada remota)", "segundos": segundos_remoto},
]))


Loaded as API: https://pbueno467-tp-integrador.hf.space/ ✔
Tipo de retorno de gradio_client: <class 'list'>
Tipo del primer elemento: <class 'dict'>
Etiqueta remota: negative
Puntaje remoto: 0.7447755336761475
Tiempo de respuesta remoto (incluye red): 2.2485 s


,forma,segundos
0,pipeline (modelo cargado),0.059260
1,gradio_client (llamada remota),2.248469


In [3]:
%pip install -r requirements-notebook.txt

  Using cached ipykernel-6.31.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached torch-2.14.1-cp313-cp313-win_amd64.whl.metadata (38 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached sentencepiece-0.2.2-cp313-cp313-win_amd64.whl.metadata (34 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached regex-2026.9.29-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached ipykernel-6.31.0-py3-none-any.whl (117 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd6


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Qué llevar al informe

Después de ejecutar el cuaderno, guardá: (1) la tabla completa de frases y resultados para cada modelo; (2) los aciertos y casos de error que realmente observaste; (3) el tipo de retorno y la forma de acceder a etiqueta y puntaje; (4) los tiempos medidos, aclarando que el local es una inferencia caliente y el remoto incluye red; (5) cualquier error real, con mensaje completo, lugar, causa y pasos de solución. No presentes los tiempos de otra computadora como propios. Antes de entregar, quitá secretos de las salidas y comprobá que el notebook se pueda reiniciar y ejecutar de principio a fin.
